# Bank Marketing Strategy — Pipeline Walkthrough

> KMeans segmentation of credit-card customers.

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH)
print(df.shape)
df.head()

(3000, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C17875,16.834929,0.454545,15.00,15.00,0.00,209.025389,0.090909,0.090909,0.000000,0.090909,1,1,7500.0,430.213001,86.959785,0.000000,11
1,C16296,540.020858,1.000000,612.23,495.61,116.62,1708.923217,0.666667,0.166667,0.500000,0.333333,10,10,2000.0,1642.068707,419.956251,0.000000,12
2,C17219,119.237712,1.000000,342.74,0.00,342.74,0.000000,1.000000,0.000000,1.000000,0.000000,0,20,2000.0,327.166041,165.207233,0.000000,12
3,C13108,894.081947,1.000000,1901.71,1853.11,48.60,206.618780,0.666667,0.666667,0.416667,0.083333,1,33,1500.0,947.130141,220.745296,0.000000,12
4,C13576,1294.145453,1.000000,3059.10,1836.98,1222.12,0.000000,1.000000,0.416667,1.000000,0.000000,0,42,7000.0,5560.033502,497.637767,0.083333,12


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
X, raw = Preprocessor().run(df)
print("customers x features:", X.shape)
X.head()

customers x features: (3000, 17)


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,16.834929,0.454545,15.00,15.00,0.00,209.025389,0.090909,0.090909,0.000000,0.090909,1,1,7500.0,430.213001,86.959785,0.000000,11
1,540.020858,1.000000,612.23,495.61,116.62,1708.923217,0.666667,0.166667,0.500000,0.333333,10,10,2000.0,1642.068707,419.956251,0.000000,12
2,119.237712,1.000000,342.74,0.00,342.74,0.000000,1.000000,0.000000,1.000000,0.000000,0,20,2000.0,327.166041,165.207233,0.000000,12
3,894.081947,1.000000,1901.71,1853.11,48.60,206.618780,0.666667,0.666667,0.416667,0.083333,1,33,1500.0,947.130141,220.745296,0.000000,12
4,1294.145453,1.000000,3059.10,1836.98,1222.12,0.000000,1.000000,0.416667,1.000000,0.000000,0,42,7000.0,5560.033502,497.637767,0.083333,12


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "trained_at": "2026-06-14T14:36:37+00:00",
  "dataset": "arjunbhasin2013/ccdata",
  "data_sha256": "29eea6389837c40fd760c927b5782563edeff55a37824598489716bdb3563be1",
  "problem": "customer segmentation (KMeans clustering)",
  "k_selection": [
    {
      "k": 3,
      "silhouette": 0.2255,
      "davies_bouldin": 1.6813,
      "calinski_harabasz": 2642.9
    },
    {
      "k": 4,
      "silhouette": 0.2124,
      "davies_bouldin": 1.6632,
      "calinski_harabasz": 2256.1
    },
    {
      "k": 5,
      "silhouette": 0.2181,
      "davies_bouldin": 1.5949,
      "calinski_harabasz": 2129.2
    },
    {
      "k": 6,
      "silhouette": 0.2179,
      "davies_bouldin": 1.4674,
      "calinski_harabasz": 2009.4
    },
    {
      "k": 7,
      "silhouette": 0.2064,
      "davies_bouldin": 1.5145,
      "calinski_harabasz": 1836.9
    },
    {
      "k": 8,
      "silhouette": 0.1955,
      "davies_bouldin": 1.7089,
      "calinski_harabasz": 1647.1
    }


## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
customer = {"BALANCE":2000,"PURCHASES":3000,"CASH_ADVANCE":0,"CREDIT_LIMIT":6000,"PAYMENTS":2500,"PURCHASES_FREQUENCY":0.9,"PRC_FULL_PAYMENT":0.3}
print("assignment:", pred.assign(customer))
print("segments:", pred.segment_table())

assignment: {'segment': 0, 'label': 'High purchases frequency', 'share_pct': 35.5, 'coords': [1.520549495720284, 0.3797723952183996], 'profile_means': {'BALANCE': 2060.89, 'PURCHASES': 2309.64, 'CASH_ADVANCE': 775.36, 'CREDIT_LIMIT': 5776.94, 'PAYMENTS': 2648.14, 'PURCHASES_FREQUENCY': 0.84, 'PRC_FULL_PAYMENT': 0.16, 'TENURE': 11.79}}
segments: [{'segment': 0, 'share_pct': 35.5, 'profile': 'High purchases frequency'}, {'segment': 1, 'share_pct': 32.4, 'profile': 'Low balance'}, {'segment': 2, 'share_pct': 32.1, 'profile': 'High cash advance, low purchases frequency'}]


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```